In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

import segmentation_models_pytorch as smp

from transformers import SegformerForSemanticSegmentation

import pandas as pd

In [14]:
IMG_SIZE = 512

BATCH_SIZE = 8
FEATURES = [32, 64, 128, 256]

LEARNING_RATE = 1e-4

EPOCHS = 10

NUM_WORKERS = 0

IN_CHANNELS = 3
OUT_CHANNELS = 1

SMOOTH = 1e-6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = {
    "U-Net": {
        "Dice": 0.7245,
        "IoU": 0.5690
    }
}

In [15]:
PROCESSED_DIR = Path("./processed")

TRAIN_DIR = PROCESSED_DIR / "train"
TRAIN_MASK_DIR = PROCESSED_DIR / "train_labels"

VAL_DIR = PROCESSED_DIR / "val"
VAL_MASK_DIR = PROCESSED_DIR / "val_labels"

TEST_DIR = PROCESSED_DIR / "test"
TEST_MASK_DIR = PROCESSED_DIR / "test_labels"

In [16]:
class RoadDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.images = sorted(self.image_dir.glob("*.png"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        mask_path = self.mask_dir / img_path.name
        image = np.array(Image.open(img_path).convert("RGB"),dtype=np.float32)
        image /= 255.0
        mask = np.array(Image.open(mask_path),dtype=np.float32)
        mask = (mask > 127).astype(np.float32)
        image = torch.tensor(image).permute(2, 0, 1)
        mask = torch.tensor(mask).unsqueeze(0)
        return image, mask

In [17]:
train_ds = RoadDataset(TRAIN_DIR,TRAIN_MASK_DIR)
val_ds = RoadDataset(VAL_DIR,VAL_MASK_DIR)
test_ds = RoadDataset(TEST_DIR,TEST_MASK_DIR)

train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
val_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_loader = DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)

In [18]:
class CONV(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

In [19]:
class BCEDiceLoss(nn.Module):

    def __init__(self, smooth=SMOOTH):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice_score = (2 * intersection + self.smooth) / (pred.sum() +target.sum() +self.smooth)
        dice_loss = 1 - dice_score
        return bce_loss + dice_loss

In [20]:
@torch.no_grad()
def dice_score(pred, target, smooth=SMOOTH):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    
    return (2 * intersection + smooth) / (pred.sum() +target.sum() +smooth)

@torch.no_grad()
def iou_score(pred, target, smooth=SMOOTH):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    union = (pred.sum() +target.sum() -intersection)
    
    return (intersection + smooth) / (union + smooth)

In [21]:
def trainer(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader,desc="Training",leave=False)
    for images, masks in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0
    pbar = tqdm(loader,desc="Validation",leave=False)
    for images, masks in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, masks)
        val_loss += loss.item()
        val_dice += dice_score(outputs,masks).item()
        val_iou += iou_score(outputs,masks).item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return (val_loss / len(loader),val_dice / len(loader),val_iou / len(loader))

In [22]:
model_unet = smp.Unet(encoder_name="resnet34",encoder_weights="imagenet",in_channels=3,classes=1).to(DEVICE)
criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(model_unet.parameters(),lr=LEARNING_RATE)
print(f"Parameters: {sum(p.numel() for p in model_unet.parameters()):,}")

Parameters: 24,436,369


In [23]:
train_losses = []
val_losses = []

val_dices = []
val_ious = []

best_dice = 0.0
best_epoch = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")
    train_loss = trainer(model_unet,train_loader,optimizer,criterion)
    val_loss, val_dice, val_iou = validate(model_unet,val_loader,criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    val_dices.append(val_dice)
    val_ious.append(val_iou)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        best_epoch = epoch + 1
        torch.save(model_unet.state_dict(),"unet_resnet34.pth")
        print(f"Saved Model")

print(f"Best Epoch: {best_epoch}")
print(f"Best Dice: {best_dice:.4f}")


Epoch [1/10]


Train Loss: 1.1757 | Val Loss: 0.9523 | Dice: 0.4843 | IoU: 0.3196
Saved Model

Epoch [2/10]


Train Loss: 0.8922 | Val Loss: 0.7710 | Dice: 0.5833 | IoU: 0.4120
Saved Model

Epoch [3/10]


Train Loss: 0.7385 | Val Loss: 0.6407 | Dice: 0.6343 | IoU: 0.4648
Saved Model

Epoch [4/10]


Train Loss: 0.6463 | Val Loss: 0.5776 | Dice: 0.6551 | IoU: 0.4874
Saved Model

Epoch [5/10]


...

Train Loss: 0.5543 | Val Loss: 0.5161 | Dice: 0.6768 | IoU: 0.5121
Saved Model

Epoch [7/10]


Train Loss: 0.5230 | Val Loss: 0.5093 | Dice: 0.6750 | IoU: 0.5097

Epoch [8/10]


Train Loss: 0.5048 | Val Loss: 0.4957 | Dice: 0.6852 | IoU: 0.5215
Saved Model

Epoch [9/10]


Train Loss: 0.4878 | Val Loss: 0.4914 | Dice: 0.6842 | IoU: 0.5203

Epoch [10/10]


Train Loss: 0.4705 | Val Loss: 0.4859 | Dice: 0.6858 | IoU: 0.5222
Saved Model
Best Epoch: 10
Best Dice: 0.6858


In [24]:
model_unet.load_state_dict(torch.load("unet_resnet34.pth", map_location=DEVICE))
model_unet.eval()

test_loss, test_dice, test_iou = validate(model_unet,test_loader,criterion)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Dice: {test_dice:.4f}")
print(f"Test IoU : {test_iou:.4f}")

results["ResNet34 U-Net"] = {
    "Dice": test_dice,
    "IoU": test_iou
}

Test Loss: 0.4058
Test Dice: 0.7190
Test IoU : 0.5627


In [25]:
model_deep = smp.DeepLabV3Plus(encoder_name="resnet34",encoder_weights="imagenet",in_channels=3,classes=1).to(DEVICE)
criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(model_deep.parameters(),lr=LEARNING_RATE)
print(f"Parameters: {sum(p.numel() for p in model_deep.parameters()):,}")

Parameters: 22,437,457


In [ ]:
train_losses = []
val_losses = []

val_dices = []
val_ious = []

best_dice = 0.0
best_epoch = 0

for epoch in range(20):
    print(f"\nEpoch [{epoch + 1}/{20}]")
    train_loss = trainer(model_deep,train_loader,optimizer,criterion)
    val_loss, val_dice, val_iou = validate(model_deep,val_loader,criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    val_dices.append(val_dice)
    val_ious.append(val_iou)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        best_epoch = epoch + 1
        torch.save(model_deep.state_dict(),"deeplab.pth")
        print(f"Saved Model")

print(f"Best Epoch: {best_epoch}")
print(f"Best Dice: {best_dice:.4f}")

In [32]:
model_deep.load_state_dict(torch.load("deeplab.pth", map_location=DEVICE))
model_deep.eval()

test_loss, test_dice, test_iou = validate(model_deep,test_loader,criterion)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Dice: {test_dice:.4f}")
print(f"Test IoU : {test_iou:.4f}")

results["Resnet-34 DeepLabV3Plus"] = {
    "Dice": test_dice,
    "IoU": test_iou
}

Test Loss: 0.5188
Test Dice: 0.6400
Test IoU : 0.4712


In [35]:
class SegFormerLoss(nn.Module):

    def __init__(self):
        super().__init__()
        self.loss_fn = BCEDiceLoss()

    def forward(self, outputs, masks):
        logits = outputs.logits
        logits = torch.nn.functional.interpolate(logits,size=masks.shape[-2:],mode="bilinear",align_corners=False)
        return self.loss_fn(logits,masks)

In [36]:
model_seg = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512",num_labels=1,ignore_mismatched_sizes=True).to(DEVICE)
criterion = SegFormerLoss()
optimizer = torch.optim.Adam(model_seg.parameters(),lr=LEARNING_RATE)
print(f"Parameters: {sum(p.numel() for p in model_seg.parameters()):,}")

[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([1])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([1, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Parameters: 3,714,401


In [37]:
def trainer_seg(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader,desc="Training",leave=False)
    for images, masks in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs,masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader)

@torch.no_grad()
def validate_seg(model, loader, criterion):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0
    pbar = tqdm(loader,desc="Validation",leave=False)
    for images, masks in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        outputs = model(pixel_values=images)
        loss = criterion(outputs,masks)
        logits = outputs.logits
        logits = torch.nn.functional.interpolate(logits,size=masks.shape[-2:],mode="bilinear",align_corners=False)
        val_loss += loss.item()
        val_dice += dice_score(logits,masks).item()
        val_iou += iou_score(logits,masks).item()

    return (val_loss / len(loader),val_dice / len(loader),val_iou / len(loader))

In [41]:
train_losses = []
val_losses = []

val_dices = []
val_ious = []

best_dice = 0.0
best_epoch = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")
    train_loss = trainer_seg(model_seg,train_loader,optimizer,criterion)
    val_loss, val_dice, val_iou = validate_seg(model_seg,val_loader,criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    val_dices.append(val_dice)
    val_ious.append(val_iou)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        best_epoch = epoch + 1
        torch.save(model_seg.state_dict(),"segformer.pth")
        print(f"Saved Model")

print(f"Best Epoch: {best_epoch}")
print(f"Best Dice: {best_dice:.4f}")


Epoch [1/10]


Train Loss: 1.0864 | Val Loss: 0.9956 | Dice: 0.4147 | IoU: 0.2617
Saved Model

Epoch [2/10]


Train Loss: 0.9389 | Val Loss: 0.8622 | Dice: 0.4683 | IoU: 0.3058
Saved Model

Epoch [3/10]


Train Loss: 0.8456 | Val Loss: 0.8049 | Dice: 0.5083 | IoU: 0.3408
Saved Model

Epoch [4/10]


Train Loss: 0.7825 | Val Loss: 0.7429 | Dice: 0.5216 | IoU: 0.3529
Saved Model

Epoch [5/10]


...

Train Loss: 0.7160 | Val Loss: 0.7009 | Dice: 0.5499 | IoU: 0.3793
Saved Model

Epoch [7/10]


Train Loss: 0.6964 | Val Loss: 0.6933 | Dice: 0.5558 | IoU: 0.3849
Saved Model

Epoch [8/10]


Train Loss: 0.6823 | Val Loss: 0.6778 | Dice: 0.5647 | IoU: 0.3935
Saved Model

Epoch [9/10]


Train Loss: 0.6687 | Val Loss: 0.6683 | Dice: 0.5656 | IoU: 0.3943
Saved Model

Epoch [10/10]


Train Loss: 0.6583 | Val Loss: 0.6579 | Dice: 0.5726 | IoU: 0.4012
Saved Model
Best Epoch: 10
Best Dice: 0.5726


In [43]:
model_seg.load_state_dict(torch.load("segformer.pth", map_location=DEVICE))
model_seg.eval()

test_loss, test_dice, test_iou = validate_seg(model_seg,test_loader,criterion)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Dice: {test_dice:.4f}")
print(f"Test IoU : {test_iou:.4f}")

results["SegformerForSemanticSegmentation"] = {
    "Dice": test_dice,
    "IoU": test_iou
}

Test Loss: 0.5726
Test Dice: 0.6015
Test IoU : 0.4306


In [ ]:
df = pd.DataFrame(results).T
df = df.sort_values(by="Dice",ascending=False)
print(df)